<h1 align='center'>🤖 SSLeech — Heroku Deployer</h1>

<center><img src='https://te.legra.ph/file/8086f391e542ed5c6a4c2.jpg' height='180' width='360' alt='SSLeech'/></center>

---
### ℹ️ Bot Details
- 🔗 **Bot Repo :** https://github.com/SunilSharmaNP/SSLeech/tree/ssleech-hk
- 🚀 **Deploy Repo :** https://github.com/SunilSharmaNP/VidLM

---
### 📋 Steps
1. **Cell 1** — Heroku Login
2. **Cell 2** — Create Heroku App
3. **Cell 3** — Fill mandatory vars → saves to MongoDB + sets Heroku config
4. **Cell 4** — Deploy to Heroku
5. **Cell 5** _(optional)_ — View logs

> **💡 After first deploy:** All further changes (TELEGRAM_API, OWNER_ID, etc.) can be done
> directly from the bot's `/botsettings` in Telegram.
> Only `BOT_TOKEN` and `DATABASE_URL` are stored in Heroku — everything else lives in MongoDB.

In [ ]:
#@title <h3>1️⃣ Heroku Login</h3>

#@markdown ---
Heroku_Email = "" #@param {type:"string"}
Heroku_API   = "" #@param {type:"string"}
#@markdown <small>🔑 Get API key: https://dashboard.heroku.com/account → **API Key**</small>
#@markdown ---

from IPython.display import HTML, clear_output, display

if not Heroku_Email or not Heroku_API:
    raise ValueError("❌ Fill in Heroku_Email and Heroku_API before running!")

!curl -s https://cli-assets.heroku.com/install.sh | sh
clear_output()

from os import path as ospath, chmod
netrc_path = ospath.expanduser("~/.netrc")
netrc_content = (
    f"machine api.heroku.com\n  login {Heroku_Email}\n  password {Heroku_API}\n"
    f"machine git.heroku.com\n  login {Heroku_Email}\n  password {Heroku_API}\n"
)
with open(netrc_path, "w") as f:
    f.write(netrc_content)
chmod(netrc_path, 0o600)

!git config --global user.email "{Heroku_Email}"
!git config --global user.name "SSLeech"

display(HTML("<b style='color:green'>✅ Heroku Login successful!</b>"))

In [ ]:
#@title <h3>2️⃣ Create Heroku App</h3>

#@markdown ---
App_Name      = "" #@param {type:"string"}
#@markdown <small>Leave blank for a random Heroku-generated name</small>
Server_Region = "us" #@param ["us", "eu"]
HK_Team_Name  = "" #@param {type:"string"}
#@markdown <small>Optional — only if deploying to a Heroku Team</small>
#@markdown ---

import re, subprocess
from IPython.display import HTML, display

if HK_Team_Name:
    cmd = f"heroku create {App_Name} --region {Server_Region} --team {HK_Team_Name}"
else:
    cmd = f"heroku create {App_Name} --region {Server_Region}"

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
output = result.stdout + result.stderr
print(output)

match = re.search(r'https://([\w-]+)\.herokuapp\.com', output)
if match:
    App_Name = match.group(1)
    display(HTML(f"<b style='color:green'>✅ App created: <code>{App_Name}</code></b>"))
elif result.returncode == 0:
    display(HTML("<b style='color:green'>✅ Done. Set App_Name manually if needed.</b>"))
else:
    display(HTML("<b style='color:red'>❌ Failed — check output above.</b>"))

In [ ]:
#@title <h3>3️⃣ Config Setup — Saves to MongoDB + Heroku</h3>

#@markdown ---
#@markdown ### 🔴 Mandatory Variables
App_Name       = "" #@param {type:"string"}
#@markdown <small>Heroku app name from Step 2</small>
BOT_TOKEN      = "" #@param {type:"string"}
#@markdown <small>From @BotFather on Telegram</small>
TELEGRAM_API   = 0  #@param {type:"integer"}
#@markdown <small>From https://my.telegram.org</small>
TELEGRAM_HASH  = "" #@param {type:"string"}
#@markdown <small>From https://my.telegram.org</small>
OWNER_ID       = 0  #@param {type:"integer"}
#@markdown <small>Your Telegram user ID — get from @userinfobot</small>
DATABASE_URL   = "" #@param {type:"string"}
#@markdown <small>MongoDB Atlas connection string (mongodb+srv://...)</small>
BASE_URL       = "" #@param {type:"string"}
#@markdown <small>Your Heroku app URL e.g. https://your-app.herokuapp.com</small>

#@markdown ---
#@markdown ### 🟡 Optional Variables
UPSTREAM_REPO   = "https://github.com/SunilSharmaNP/SSLeech" #@param {type:"string"}
UPSTREAM_BRANCH = "ssleech-hk" #@param {type:"string"}
TIMEZONE        = "Asia/Kolkata" #@param {type:"string"}
#@markdown ---

from IPython.display import HTML, display
import subprocess, sys

# ── Validate ─────────────────────────────────────────────────────────────────
missing = [k for k, v in {
    "App_Name": App_Name, "BOT_TOKEN": BOT_TOKEN, "TELEGRAM_API": TELEGRAM_API,
    "TELEGRAM_HASH": TELEGRAM_HASH, "OWNER_ID": OWNER_ID,
    "DATABASE_URL": DATABASE_URL, "BASE_URL": BASE_URL,
}.items() if not v]
if missing:
    raise ValueError(f"❌ Fill these mandatory fields: {', '.join(missing)}")

# ── Install pymongo ───────────────────────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "install", "pymongo[srv]", "-q"])

# ── Step A: Save ALL vars to MongoDB ─────────────────────────────────────────
# KEY FIX: vars saved here so update.py loads them on every Heroku restart.
# Heroku Config Vars only need BOT_TOKEN + DATABASE_URL — nothing else.
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

BOT_ID = BOT_TOKEN.split(":", 1)[0]
config_doc = {
    "_id":             BOT_ID,
    "BOT_TOKEN":       BOT_TOKEN,
    "TELEGRAM_API":    str(TELEGRAM_API),
    "TELEGRAM_HASH":   TELEGRAM_HASH,
    "OWNER_ID":        str(OWNER_ID),
    "DATABASE_URL":    DATABASE_URL,
    "BASE_URL":        BASE_URL,
    "UPSTREAM_REPO":   UPSTREAM_REPO,
    "UPSTREAM_BRANCH": UPSTREAM_BRANCH,
    "TIMEZONE":        TIMEZONE,
}

try:
    conn = MongoClient(DATABASE_URL, server_api=ServerApi("1"), serverSelectionTimeoutMS=10000)
    conn.wzmlx.settings.config.replace_one({"_id": BOT_ID}, config_doc, upsert=True)
    conn.close()
    display(HTML("<b style='color:green'>✅ Config saved to MongoDB!</b>"))
except Exception as e:
    display(HTML(f"<b style='color:red'>❌ MongoDB error: {e}</b>"))
    raise

# ── Step B: Set ONLY BOT_TOKEN + DATABASE_URL in Heroku Config Vars ──────────
var_str = f"BOT_TOKEN={BOT_TOKEN} DATABASE_URL={DATABASE_URL}"
result  = subprocess.run(f"heroku config:set {var_str} --app {App_Name}",
                         shell=True, capture_output=True, text=True)
if result.returncode == 0:
    display(HTML("<b style='color:green'>✅ Heroku Config Vars set (BOT_TOKEN + DATABASE_URL only)</b>"))
else:
    display(HTML(f"<b style='color:orange'>⚠️ Heroku config:set issue: {result.stderr}</b>"))

display(HTML("""
<hr>
<b>ℹ️ Summary:</b><br>
• MongoDB ✅ — TELEGRAM_API, TELEGRAM_HASH, OWNER_ID etc. saved<br>
• Heroku ✅ — Only BOT_TOKEN + DATABASE_URL in Config Vars<br>
• On every restart, update.py loads everything from MongoDB automatically<br>
• Use /botsettings in Telegram to change any setting — no Heroku access needed
"""))

In [ ]:
#@title <h3>4️⃣ Deploy to Heroku</h3>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

import os, subprocess, sys
from IPython.display import HTML, display

if not App_Name:
    raise ValueError("❌ Set App_Name (same as Step 2 & 3)")

# Clone this VidLM deploy repo fresh
deploy_dir = App_Name
if os.path.isdir(deploy_dir):
    os.system(f"rm -rf {deploy_dir}")

subprocess.run("git clone https://github.com/SunilSharmaNP/VidLM " + deploy_dir,
               shell=True, check=True)
os.chdir(deploy_dir)

# Remove deploy-only files before push
for f in ["README.md", "ssleech_hk_deploy.ipynb"]:
    if os.path.isfile(f):
        os.remove(f)

# Write config.env from MongoDB (bootstrap for the very first Heroku build)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "pymongo[srv]", "-q"])
    from pymongo.mongo_client import MongoClient as _MC
    from pymongo.server_api import ServerApi as _SA
    _bid  = BOT_TOKEN.split(":", 1)[0]
    _conn = _MC(DATABASE_URL, server_api=_SA("1"), serverSelectionTimeoutMS=8000)
    _doc  = _conn.wzmlx.settings.config.find_one({"_id": _bid})
    _conn.close()
    if _doc:
        with open("config.env", "w") as _f:
            for _k, _v in _doc.items():
                if _k == "_id" or _v is None:
                    continue
                _vs = str(_v).strip()
                if _vs and _vs != "None":
                    _f.write(f'{_k}="{_vs}"\n')
        display(HTML("<b style='color:green'>✅ config.env written from MongoDB</b>"))
    else:
        display(HTML("<b style='color:orange'>⚠️ No MongoDB doc found — run Step 3 first!</b>"))
except Exception as _e:
    display(HTML(f"<b style='color:orange'>⚠️ config.env skipped: {_e}</b>"))

# Push to Heroku
os.system("git add . -f")
os.system("git commit -m 'SSLeech Deploy' --allow-empty -q")
os.system(f"heroku git:remote -a {App_Name}")
os.system("git push heroku main -f")

os.chdir("..")
display(HTML(f"""
<hr>
<b style='color:green'>🚀 Deployed!</b><br>
App URL: <a href='https://{App_Name}.herokuapp.com' target='_blank'>https://{App_Name}.herokuapp.com</a><br>
<small>View logs in Cell 5 below.</small>
"""))

In [ ]:
#@title <h3>➕ Deploy Multiple Apps</h3>

#@markdown ---
App_Names = "" #@param {type:"string"}
#@markdown <small>Space-separated: <code>bot1 bot2 bot3</code> — Run Step 3 for each app first</small>
#@markdown ---

import os, subprocess
from IPython.display import HTML, display

for app in App_Names.split():
    display(HTML(f"<hr><b>Deploying: {app}</b>"))
    if not os.path.isdir(app):
        subprocess.run(f"git clone https://github.com/SunilSharmaNP/VidLM {app}", shell=True)
    os.chdir(app)
    os.system("git add . -f")
    os.system("git commit -m 'SSLeech Deploy' --allow-empty -q")
    os.system(f"heroku git:remote -a {app}")
    os.system("git push heroku main -f")
    os.chdir("..")
    display(HTML(f"<b style='color:green'>✅ {app} deployed!</b>"))

In [ ]:
#@title <h3>5️⃣ View Heroku Logs</h3>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku logs -t -a $App_Name

In [ ]:
#@title <h3>🔄 Restart App</h3>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku restart -a $App_Name

from IPython.display import HTML, display
display(HTML(f"<b style='color:green'>✅ {App_Name} restarting...</b>"))

In [ ]:
#@title <h3>🚪 Heroku Logout</h3>
!heroku logout